In [3]:
from pydantic import BaseModel

# 1. Define the parameters for a single generator
class Generator(BaseModel):
    name: str
    cost_per_mw: float
    max_capacity_mw: float

# 2. Define the overall problem structure
class DispatchProblem(BaseModel):
    objective_type: str  # e.g., "minimize cost"
    total_demand_mw: float
    generators: list[Generator]

# 3. Create a dummy object to test if Pydantic is working
dummy_data = DispatchProblem(
    objective_type="minimize cost",
    total_demand_mw=150.0,
    generators=[
        Generator(name="Gen_1", cost_per_mw=20.0, max_capacity_mw=100.0),
        Generator(name="Gen_2", cost_per_mw=30.0, max_capacity_mw=120.0)
    ]
)

print(f"Environment Success! Schema defined. Grid demands {dummy_data.total_demand_mw} MW.")

Environment Success! Schema defined. Grid demands 150.0 MW.


In [52]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Load the hidden API key from your .env file
load_dotenv()

# 2. Initialize the Gemini client 
client = genai.Client()

# 3. Define the plain-text problem (Unstructured Data)
prompt_text = """
The grid operator needs to meet a total demand of 220 MW. 
There are three generators available. Generator 1 costs $20 per MW 
to run and has a maximum capacity of 100 MW. Generator 2 costs $30 per MW 
to run and has a maximum capacity of 120 MW. Generator 3 is a cheap solar plant that costs $10 per MW 
but only has a maximum capacity of 50 MW. Minimize the total cost.
"""

# 4. Ask Gemini to read the text and fill out the Pydantic form
print("Sending text to Gemini...")
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt_text,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=DispatchProblem,
        temperature=0.0, # We set this to 0 because we want exact facts, not creativity
    ),
)

# 5. Convert the JSON response back into our Pydantic object
extracted_data = DispatchProblem.model_validate_json(response.text)

print("\n--- Extraction Success! ---")
print(f"Objective: {extracted_data.objective_type}")
print(f"Demand: {extracted_data.total_demand_mw} MW")
for gen in extracted_data.generators:
    print(f"Found {gen.name}: Max {gen.max_capacity_mw}MW, Cost ${gen.cost_per_mw}/MW")



Sending text to Gemini...

--- Extraction Success! ---
Objective: minimize_cost
Demand: 220.0 MW
Found Generator 1: Max 100.0MW, Cost $20.0/MW
Found Generator 2: Max 120.0MW, Cost $30.0/MW
Found Generator 3: Max 50.0MW, Cost $10.0/MW


In [53]:
print(extracted_data)

objective_type='minimize_cost' total_demand_mw=220.0 generators=[Generator(name='Generator 1', cost_per_mw=20.0, max_capacity_mw=100.0), Generator(name='Generator 2', cost_per_mw=30.0, max_capacity_mw=120.0), Generator(name='Generator 3', cost_per_mw=10.0, max_capacity_mw=50.0)]


In [56]:
import pyomo.environ as pyo
model = pyo.ConcreteModel()
gen_names = [g.name for g in extracted_data.generators]
print(gen_names)
model.generator = pyo.Set(initialize = gen_names)
model.total_demand_mw = pyo.Param(initialize = extracted_data.total_demand_mw)
model.gen_power = pyo.Var(model.generator)

costs = {g.name: g.cost_per_mw for g in extracted_data.generators}
model.cost = pyo.Param(model.generator, initialize=costs)
max_capacity = {g.name: g.max_capacity_mw for g in extracted_data.generators}
model.max_capicity = pyo.Param(model.generator, initialize=max_capacity)

def max_rule(model, g):
    return model.gen_power[g] <= model.max_capicity[g]
model.c1 = pyo.Constraint(expr = sum(model.gen_power[g] for g in model.generator)== model.total_demand_mw)
model.c2 = pyo.Constraint(model.generator, rule = max_rule)

model.f1 = pyo.Objective(expr = sum(model.cost[g] * model.gen_power[g] for g in model.generator), sense=pyo.minimize)

solver = pyo.SolverFactory('glpk')
results = solver.solve(model)
for g in model.generator:
    print(f"{g}: {pyo.value(model.gen_power[g])} MW")



['Generator 1', 'Generator 2', 'Generator 3']
Generator 1: 100.0 MW
Generator 2: 70.0 MW
Generator 3: 50.0 MW
